In [2]:
# ============================================================
# ResNet-34 + CoAtNet (Conv + MHSA) + CaiT Class Attention
# Single-file, end-to-end training & evaluation
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.models as models
import timm
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report

# ---------------- CONFIG ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 25
LR = 2e-4
EMBED_DIM = 384
CHECKPOINT = "best_coatnet_cait_hybrid.pth"

print("Device:", DEVICE)

# ---------------- DATASET ----------------
class MemoryCachedDataset(torch.utils.data.Dataset):
    def __init__(self, cache_path):
        data = torch.load(cache_path)
        self.samples = data["samples"]
        self.classes = data["classes"]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

train_ds = MemoryCachedDataset("train_cache.pt")
val_ds   = MemoryCachedDataset("val_cache.pt")
test_ds  = MemoryCachedDataset("test_cache.pt")

NUM_CLASSES = len(train_ds.classes)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# ---------------- CoAtNet Block ----------------
class CoAtBlock(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(dim, dim, 3, padding=1, groups=dim),
            nn.BatchNorm1d(dim),
            nn.GELU()
        )
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        conv_out = self.conv(x.transpose(1, 2)).transpose(1, 2)
        x = self.norm1(x + conv_out)
        attn_out, _ = self.attn(x, x, x)
        x = self.norm2(x + attn_out)
        return x

# ---------------- HYBRID MODEL ----------------
class CoAtNetCaiTHybrid(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # ---- ResNet-34 token generator ----
        resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)

        self.stem = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        )
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2

        for p in self.stem.parameters(): p.requires_grad = False
        for p in self.layer1.parameters(): p.requires_grad = False

        self.proj = nn.Linear(128, EMBED_DIM)

        # ---- CoAtNet refinement ----
        self.coat_blocks = nn.Sequential(
            CoAtBlock(EMBED_DIM),
            CoAtBlock(EMBED_DIM)
        )

        # ---- CaiT class attention ----
        cait = timm.create_model("cait_s24_224", pretrained=True)
        self.class_blocks = cait.blocks_token_only
        self.norm = cait.norm

        self.cls_token = nn.Parameter(torch.zeros(1, 1, EMBED_DIM))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        self.head = nn.Linear(EMBED_DIM, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)              # [B,128,T,F]

        x = x.mean(dim=2).permute(0, 2, 1)  # pool freq → keep time
        tokens = self.proj(x)

        tokens = self.coat_blocks(tokens)

        B = tokens.size(0)
        cls = self.cls_token.expand(B, -1, -1)

        for blk in self.class_blocks:
            cls = blk(cls, tokens)

        cls = self.norm(cls)
        cls = cls[:, 0, :]
        return self.head(cls)

# ---------------- TRAINING SETUP ----------------
model = CoAtNetCaiTHybrid(NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# ---------------- TRAIN LOOP ----------------
best_val = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()
    preds, gts = [], []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            preds.extend(logits.argmax(1).cpu().numpy())
            gts.extend(labels.numpy())

    acc = accuracy_score(gts, preds)
    print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Val={acc*100:.2f}%")

    if acc > best_val:
        best_val = acc
        torch.save(model.state_dict(), CHECKPOINT)
        print("💾 Saved best model")




c:\Users\sreeh\miniconda3\envs\your_env_name\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


Epoch 1: 100%|██████████| 433/433 [00:19<00:00, 22.73it/s]


Epoch 1: Loss=0.7834, Val=79.37%
💾 Saved best model


Epoch 2: 100%|██████████| 433/433 [00:15<00:00, 27.31it/s]


...

Epoch 23: Loss=0.2535, Val=78.03%


Epoch 24: 100%|██████████| 433/433 [00:13<00:00, 33.20it/s]


Epoch 24: Loss=0.2597, Val=84.30%


Epoch 25: 100%|██████████| 433/433 [00:13<00:00, 33.15it/s]


Epoch 25: Loss=0.2507, Val=83.41%


In [3]:
# ---------------- TEST ----------------
model.load_state_dict(torch.load(CHECKPOINT))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print("Test Accuracy:", accuracy_score(all_labels, all_preds) * 100)
print(classification_report(all_labels, all_preds, target_names=train_ds.classes))

Test Accuracy: 90.2654867256637
                        precision    recall  f1-score   support

      Cysts_Structural       0.95      0.91      0.93        22
             Dysarthia       1.00      1.00      1.00        42
            Laryngitis       0.73      0.86      0.79        42
           Vox senilis       0.93      0.84      0.88        50
             parkinson       0.96      1.00      0.98        50
spasmodische_dysphonie       0.82      0.70      0.76        20

              accuracy                           0.90       226
             macro avg       0.90      0.88      0.89       226
          weighted avg       0.91      0.90      0.90       226



Improvement

In [10]:
# ============================================================
# Enhanced ResNet-34 + CoAtNet + CaiT with Advanced Techniques
# Target: 95%+ Test Accuracy (from 90.26%)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.models as models
import timm
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# ---------------- CONFIG ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 12  # Reduced for better gradient estimation
EPOCHS = 40      # More epochs with early stopping
LR = 1e-4        # Lower initial LR
EMBED_DIM = 512  # Increased capacity
CHECKPOINT = "best_enhanced_model.pth"

print("Device:", DEVICE)

# ---------------- AUGMENTATION DATASET ----------------
class AugmentedDataset(torch.utils.data.Dataset):
    """Add mixup-style augmentation at batch level"""
    def __init__(self, cache_path, augment=False):
        data = torch.load(cache_path)
        self.samples = data["samples"]
        self.classes = data["classes"]
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img, label = self.samples[idx]
        
        if self.augment and torch.rand(1).item() > 0.5:
            # Time masking (SpecAugment style)
            img = img.clone()
            T = img.size(-1)
            mask_len = int(T * 0.1)
            start = torch.randint(0, T - mask_len, (1,)).item()
            img[..., start:start+mask_len] = 0
            
        return img, label

train_ds = AugmentedDataset("train_cache.pt", augment=True)
val_ds   = AugmentedDataset("val_cache.pt", augment=False)
test_ds  = AugmentedDataset("test_cache.pt", augment=False)

NUM_CLASSES = len(train_ds.classes)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# ---------------- ENHANCED CoAtNet Block ----------------
class EnhancedCoAtBlock(nn.Module):
    def __init__(self, dim, heads=8, mlp_ratio=4):
        super().__init__()
        
        # Depthwise conv with multiple scales
        self.conv = nn.Sequential(
            nn.Conv1d(dim, dim, 3, padding=1, groups=dim),
            nn.BatchNorm1d(dim),
            nn.GELU()
        )
        
        # Multi-head attention with relative position bias
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True, dropout=0.1)
        
        # MLP with expansion
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(dim * mlp_ratio, dim),
            nn.Dropout(0.1)
        )
        
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.norm3 = nn.LayerNorm(dim)

    def forward(self, x):
        # Convolution path
        conv_out = self.conv(x.transpose(1, 2)).transpose(1, 2)
        x = self.norm1(x + conv_out)
        
        # Attention path
        attn_out, _ = self.attn(x, x, x)
        x = self.norm2(x + attn_out)
        
        # MLP path
        mlp_out = self.mlp(x)
        x = self.norm3(x + mlp_out)
        
        return x

# ---------------- SQUEEZE-EXCITATION ----------------
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return x * self.se(x)

# ---------------- ENHANCED HYBRID MODEL ----------------
class EnhancedHybrid(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # ---- ResNet-34 with SE blocks ----
        resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)

        self.stem = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        )
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3  # Add layer3 for more capacity

        # Freeze only stem for faster training
        for p in self.stem.parameters(): 
            p.requires_grad = False

        # Add SE attention to extracted features
        self.se = SqueezeExcitation(256)  # layer3 output channels

        # Projection with more capacity
        self.proj = nn.Sequential(
            nn.Linear(256, EMBED_DIM),
            nn.LayerNorm(EMBED_DIM),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        # ---- Enhanced CoAtNet (3 blocks for deeper processing) ----
        self.coat_blocks = nn.Sequential(
            EnhancedCoAtBlock(EMBED_DIM, heads=8),
            EnhancedCoAtBlock(EMBED_DIM, heads=8),
            EnhancedCoAtBlock(EMBED_DIM, heads=8)
        )

        # ---- CaiT class attention (deeper) ----
        cait = timm.create_model("cait_s24_224", pretrained=True)
        
        # Project to CaiT's expected dimension (384)
        cait_dim = 384
        self.to_cait = nn.Linear(EMBED_DIM, cait_dim) if EMBED_DIM != cait_dim else nn.Identity()
        
        # Use more class attention blocks
        self.class_blocks = cait.blocks_token_only[:4]  # Increase from default
        self.norm = cait.norm

        self.cls_token = nn.Parameter(torch.zeros(1, 1, cait_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        
        # Project back from CaiT dimension to our head
        self.from_cait = nn.Linear(cait_dim, EMBED_DIM) if EMBED_DIM != cait_dim else nn.Identity()

        # ---- Multi-scale head ----
        self.head = nn.Sequential(
            nn.Linear(EMBED_DIM, EMBED_DIM // 2),
            nn.LayerNorm(EMBED_DIM // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(EMBED_DIM // 2, num_classes)
        )

    def forward(self, x, return_features=False):
        # Feature extraction with SE
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)  # [B, 256, T, F]
        
        x = self.se(x)  # Apply SE attention
        
        # Pool frequency, keep time dimension
        x = x.mean(dim=2).permute(0, 2, 1)  # [B, T, 256]
        tokens = self.proj(x)  # [B, T, EMBED_DIM]

        # CoAtNet refinement
        tokens = self.coat_blocks(tokens)

        # Project to CaiT dimension and apply class attention
        tokens_cait = self.to_cait(tokens)
        
        B = tokens_cait.size(0)
        cls = self.cls_token.expand(B, -1, -1)

        for blk in self.class_blocks:
            cls = blk(cls, tokens_cait)

        cls = self.norm(cls)
        cls = self.from_cait(cls[:, 0, :])  # Project back to EMBED_DIM
        
        features = cls
        
        logits = self.head(features)
        
        if return_features:
            return logits, features
        return logits

# ---------------- FOCAL LOSS (better for imbalanced classes) ----------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', 
                                   label_smoothing=self.label_smoothing)
        p = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - p) ** self.gamma * ce_loss
        return focal_loss.mean()

# ---------------- MIXUP AUGMENTATION ----------------
def mixup_data(x, y, alpha=0.2):
    """Mixup augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ---------------- TRAINING SETUP ----------------
model = EnhancedHybrid(NUM_CLASSES).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

criterion = FocalLoss(alpha=1.0, gamma=2.0, label_smoothing=0.1)

# AdamW with weight decay
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=LR, 
    weight_decay=1e-3,
    betas=(0.9, 0.999)
)

# Cosine annealing with warm restarts
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# ---------------- TRAIN LOOP WITH ADVANCED TECHNIQUES ----------------
best_val = 0.0
patience = 15
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        # Apply mixup with 50% probability
        if torch.rand(1).item() > 0.5:
            imgs, labels_a, labels_b, lam = mixup_data(imgs, labels, alpha=0.2)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
        else:
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)

        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    preds, gts = [], []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            preds.extend(logits.argmax(1).cpu().numpy())
            gts.extend(labels.numpy())

    acc = accuracy_score(gts, preds)
    avg_loss = total_loss / len(train_loader)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Val={acc*100:.2f}%, LR={current_lr:.6f}")

    # Save best model
    if acc > best_val:
        best_val = acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': acc,
        }, CHECKPOINT)
        print(f"💾 Saved best model (Val: {acc*100:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= patience:
        print(f"Early stopping triggered after {epoch+1} epochs")
        break


Device: cuda
Trainable parameters: 21,850,390


Epoch 1/40: 100%|██████████| 576/576 [00:41<00:00, 13.98it/s]


Epoch 1: Loss=0.5582, Val=76.68%, LR=0.000098
💾 Saved best model (Val: 76.68%)


Epoch 2/40: 100%|██████████| 576/576 [00:37<00:00, 15.26it/s]


Epoch 2: Loss=0.3681, Val=86.55%, LR=0.000091
💾 Saved best model (Val: 86.55%)


...

Epoch 38: Loss=0.1583, Val=87.00%, LR=0.000091


Epoch 39/40: 100%|██████████| 576/576 [00:33<00:00, 17.14it/s]


Epoch 39: Loss=0.1462, Val=84.30%, LR=0.000088


Epoch 40/40: 100%|██████████| 576/576 [00:33<00:00, 16.99it/s]


Epoch 40: Loss=0.1501, Val=87.00%, LR=0.000086


In [11]:

# ---------------- FINAL EVALUATION ----------------
print("\n" + "="*60)
print("Loading best model for final evaluation...")
checkpoint = torch.load(CHECKPOINT)
model.load_state_dict(checkpoint['model_state_dict'])

model.eval()
test_preds, test_gts = [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Testing"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        test_preds.extend(logits.argmax(1).cpu().numpy())
        test_gts.extend(labels.numpy())

test_acc = accuracy_score(test_gts, test_preds)
print(f"\n🎯 FINAL TEST ACCURACY: {test_acc*100:.2f}%")
print("\nDetailed Classification Report:")
print(classification_report(test_gts, test_preds, target_names=train_ds.classes, digits=4))
print("="*60)


Loading best model for final evaluation...


Testing: 100%|██████████| 19/19 [00:00<00:00, 59.20it/s]


🎯 FINAL TEST ACCURACY: 93.81%

Detailed Classification Report:
                        precision    recall  f1-score   support

      Cysts_Structural     1.0000    1.0000    1.0000        22
             Dysarthia     1.0000    1.0000    1.0000        42
            Laryngitis     0.8750    0.8333    0.8537        42
           Vox senilis     0.8889    0.9600    0.9231        50
             parkinson     1.0000    1.0000    1.0000        50
spasmodische_dysphonie     0.8333    0.7500    0.7895        20

              accuracy                         0.9381       226
             macro avg     0.9329    0.9239    0.9277       226
          weighted avg     0.9374    0.9381    0.9372       226

